# MNIST Unity Cloud PC / Quest 3S Data and Benchmarking

This notebook documents the **cloud-assisted MNIST inference workflow** used with both the Windows PC Unity application and the Meta Quest 3S Unity application.

The cloud implementation extends the existing local PC and Quest applications. It does not use separate cloud scenes. Instead, cloud-specific scripts replace local Unity Inference Engine execution with HTTP requests to the deployed Google Cloud Run inference service.

This notebook documents:

- the cloud client architecture;
- the shared request and response format;
- PC cloud validation and benchmarks;
- PC editor and standalone-build evidence;
- PC memory observations;
- Quest cloud tethered testing;
- Quest cloud untethered testing;
- OVR Metrics logs;
- camera-based cloud benchmarks;
- power and Internet-dependency evidence; and
- the repository files that support reproduction.

The corresponding code branch contains the four cloud-specific Unity scripts.

## 1. Relationship to the Local Implementations

The cloud clients build directly on the previously documented local implementations.

### Local PC branch

Provides the existing Windows Unity scene, webcam pipeline, Canvas, MNIST test assets, and baseline benchmark structure.

### Local Quest 3S branch

Provides `SecondScene.unity`, the Meta XR rig, Quest camera access, XR Canvas, poke/ray interaction, and Quest deployment configuration.

### Cloud branch

Changes the inference path from:

```text
local Unity Inference Engine
```

to:

```text
HTTP request
    ↓
Google Cloud Run
    ↓
prediction response
```

No separate cloud-specific Unity scenes were created.

## 2. Cloud-Specific Unity Scripts

The cloud code branch contains:

```text
CloudMNISTSingleTest.cs
CloudMNISTBenchmark.cs
CloudCameraMNISTInference.cs
QuestCloudCameraMNISTInference.cs
```

Their roles are:

| Script | Role |
|---|---|
| `CloudMNISTSingleTest.cs` | Validate one MNIST image against the Cloud Run endpoint |
| `CloudMNISTBenchmark.cs` | Run the controlled 1,000-image cloud benchmark |
| `CloudCameraMNISTInference.cs` | PC webcam → preprocessing → Cloud Run |
| `QuestCloudCameraMNISTInference.cs` | Quest passthrough camera → preprocessing → Cloud Run |

The existing PC and Quest scene structures were reused.

## 3. Cloud Run Endpoint

The Unity clients reference the deployed prediction endpoint:

```text
https://mnist-cloud-inference-221036165238.us-east1.run.app/predict
```

The endpoint is exposed as a Unity script field so that it can be changed if the service is redeployed.

The cloud clients use HTTP `POST` requests with:

```text
Content-Type: application/octet-stream
```

## 4. Cloud Request Representation

The original Unity MNIST test data stores each pixel as a 32-bit floating-point value in the range:

```text
0.0–1.0
```

For cloud transmission, each pixel is converted to:

```text
0–255 byte
```

A single image is therefore transmitted as:

```text
784 bytes
```

rather than the original 784 floating-point values.

The Cloud Run service returns a JSON prediction in the form:

```json
{"prediction":N}
```

## 5. Timing Boundary

Cloud latency is measured differently from local inference latency.

### Local inference

The local benchmark measures model execution and synchronous output readback inside Unity.

### Cloud inference

Cloud timing begins immediately before the HTTP request is sent and ends when the complete response is available.

Conceptually:

```text
request transmission
    ↓
network path
    ↓
Cloud Run processing
    ↓
response transmission
    ↓
Unity receives response
```

The result is therefore a **round-trip cloud inference measurement**, not isolated neural-network execution time.

## 6. Controlled Cloud Benchmark

`CloudMNISTBenchmark.cs` evaluates:

```text
1000 MNIST test images
```

Each image is sent as a separate HTTP request.

For each request, the script records:

- latency;
- returned prediction;
- expected label; and
- request success/error status.

At completion, the benchmark calculates:

- correct predictions;
- request errors;
- accuracy;
- mean latency;
- median latency;
- P95 latency;
- minimum latency;
- maximum latency;
- total elapsed time; and
- throughput in requests per second.

The benchmark also writes a CSV containing summary and per-image results.

## 7. Camera Cloud Benchmark Settings

The PC and Quest camera cloud scripts use fewer repetitions than the local camera benchmarks because every inference requires a network request.

The camera benchmark settings are:

```text
2 warm-up requests
100 measured requests
```

The cloud camera implementation supports:

- single-frame classification;
- frozen-camera benchmark;
- end-to-end camera benchmark; and
- live camera throughput.

## 8. PC Cloud Workflow

The PC cloud implementation reuses the existing desktop webcam and Canvas.

The pipeline is:

```text
Laptop webcam
    ↓
FPGA-matched camera preprocessing
    ↓
28×28 image
    ↓
byte conversion
    ↓
HTTP request
    ↓
Cloud Run prediction
    ↓
Unity UI
```

`CloudCameraMNISTInference.cs` replaces the local PC inference worker while preserving the camera acquisition and preprocessing structure.

## 9. PC Controlled Benchmark Evidence

The repository contains controlled PC cloud benchmark screenshots from the Unity Editor and Windows standalone build.

Relevant files include:

```text
cloud_pc_accuracy_latency_throughput.png
cloud_pc_accuracy_latency_throughput2.png
cloud_pc_Accuracy_latency_throughput_build.png
```

The controlled benchmark provides cloud:

- accuracy;
- latency distribution; and
- request throughput.

Editor and standalone-build captures are preserved separately because they document different execution contexts.

## 10. PC Single Classification Evidence

The PC cloud client was also validated with a single-image cloud request.

Supporting evidence includes:

```text
cloud_pc_classification.png
```

This verifies:

- client-to-cloud connectivity;
- correct request formatting;
- JSON response parsing; and
- return of a usable prediction to the Unity application.

## 11. PC Frozen-Camera Benchmark Evidence

The PC frozen-camera cloud benchmark holds the already processed 28×28 camera image constant and repeatedly sends that image to the Cloud Run service.

Supporting files include:

```text
cloud_pc_frozen_benchmark.png
cloud_pc_frozen_benchmark_build.png
```

The frozen-camera benchmark excludes repeated camera acquisition/preprocessing from the measured request sequence and focuses on repeated cloud requests using a camera-derived input.

## 12. PC End-to-End Camera Evidence

The PC end-to-end cloud benchmark includes:

```text
camera image acquisition
    ↓
coordinate mapping
    ↓
28×28 sampling
    ↓
grayscale conversion
    ↓
binary threshold
    ↓
request-body creation
    ↓
Cloud Run request
    ↓
prediction response
```

Supporting files include:

```text
cloud_pc_end_to_end.png
cloud_pc_end_to_end_build.png
```

This benchmark measures the complete application path rather than cloud request latency alone.

## 13. PC Live Classification and Throughput

Supporting PC cloud camera files include:

```text
cloud_pc_live_classification.png
cloud_pc_live_classification_build.png
cloud_pc_live_throughput.png
cloud_pc_live_throughput_build.png
```

Live classification demonstrates normal camera-to-cloud operation.

Live throughput measures completed cloud predictions using newly available camera frames over elapsed real time.

This application-level throughput is distinct from the 1,000-image controlled request benchmark.

## 14. PC Memory Evidence

The repository contains a sequence of Windows memory observations:

```text
cloud_pc_memory1.png
cloud_pc_memory2.png
cloud_pc_memory3.png
cloud_pc_memory4.png
cloud_pc_memory5.png
cloud_pc_memory6.png
cloud_pc_memory7.png
```

These screenshots document Windows process/resource behavior during cloud-client execution.

Because cloud inference occurs remotely, these observations characterize the Unity client and its camera/network workload rather than local neural-network model execution.

## 15. PC Power and Internet Evidence

The repository includes video evidence related to the PC cloud configuration:

```text
Cloud Power Unity10-22-202606-36.mp4
Cloud Power 2 Unity10-26-202608-11.mp4
Unity Cloud Internet10-22-202610-13.mp4
```

The power videos document whole-system PC power observations during cloud-assisted use.

`Unity Cloud Internet10-22-202610-13.mp4` documents the network-dependent nature of the cloud configuration.

Unlike the local PC implementation, successful cloud inference requires access to the remote Cloud Run endpoint.

## 16. Quest Cloud Workflow

The Quest cloud implementation reuses the Quest `SecondScene` and existing Meta XR configuration.

The camera path becomes:

```text
Quest passthrough camera
    ↓
same FPGA-matched preprocessing
    ↓
28×28 image
    ↓
byte conversion
    ↓
HTTP request
    ↓
Cloud Run
    ↓
prediction response
    ↓
XR Canvas
```

The Quest-specific cloud script is:

```text
QuestCloudCameraMNISTInference.cs
```

The local Quest neural-network worker is not used for cloud inference.

## 17. Quest Connection Conditions

Quest cloud testing was performed under two connection conditions:

```text
tethered
untethered
```

The distinction is important because the cloud implementation depends on network communication.

### Tethered evidence

The `.png` Quest cloud screenshots were captured through casting and therefore document the **tethered** condition.

### Untethered evidence

The `.jpg` Quest cloud screenshots document the **untethered** condition.

The two conditions are preserved separately for comparison.

## 18. Quest Tethered Controlled Benchmark

Tethered controlled-benchmark evidence includes:

```text
quest_cloud_accuracy_latency_throughput.png
quest_cloud_accuracy_latency_throughput2.png
```

Additional progress captures include:

```text
quest_cloud_benchmark.png
quest_cloud_benchmark2.png
quest_cloud_benchmark3.png
quest_cloud_benchmark4.png
```

The final controlled benchmark screenshots are the primary evidence; progress screenshots document the benchmark while it was running.

Representative tethered results show:

```text
Images:   1000
Correct:  957
Errors:   0
Accuracy: 95.70%
```

The repeated final runs provide separate latency/throughput observations under the tethered condition.

## 19. Quest Tethered Classification

Tethered Quest cloud classification evidence includes:

```text
quest_cloud_frozen_classification.png
quest_cloud_live_classification.png
```

These screenshots demonstrate:

- preprocessing of a live Quest camera image;
- cloud request execution;
- returned prediction; and
- round-trip latency displayed inside the Quest application.

## 20. Quest Tethered Frozen-Camera Benchmark

The tethered frozen-camera cloud benchmark is documented by:

```text
quest_cloud_frozen_benchmark.png
```

Representative result:

```text
Runs:   100
Mean:   ~53.26 ms
Median: ~48.10 ms
P95:    ~73.52 ms
```

The exact values should be taken from the archived image when used in a final results table.

## 21. Quest Tethered End-to-End Camera Benchmark

Tethered end-to-end evidence includes:

```text
quest_cloud_end_to_end.png
quest_cloud_end_to_end2.png
```

The final result screenshot is the primary benchmark evidence, while the second file may document the benchmark in progress.

Representative final result:

```text
Runs:   100
Mean:   ~79.20 ms
Median: ~76.92 ms
P95:    ~93.01 ms
```

## 22. Quest Tethered Live Throughput

Tethered live-throughput evidence includes:

```text
quest_cloud_live_throughput.png
quest_cloud_live_throughput2.png
```

One screenshot documents the test while running; the other preserves the completed result.

Representative completed result:

```text
Completed requests: 182
Time:               ~10.02 s
Throughput:         ~18.16 predictions/s
```

## 23. Quest Tethered OVR Metrics Logs

The tethered Quest cloud telemetry logs are:

```text
quest_cloud_ovrmetrics.csv
quest_cloud_ovrmetrics2.csv
quest_cloud_ovrmetrics3.csv
quest_cloud_ovrmetrics4.csv
```

These sessions provide device-side context during the cast/tethered cloud test period.

They should be treated as session telemetry rather than assigned to a specific benchmark unless the collection timing clearly establishes that mapping.

## 24. Quest Untethered Controlled Benchmark

Untethered controlled-benchmark evidence includes:

```text
quest_cloud_untet_accuracy_latency_throughput.jpg
```

A progress capture is also included:

```text
quest_cloud_untet_benchmark.jpg
```

Representative final result:

```text
Images:     1000
Correct:    957
Errors:     0
Accuracy:   95.70%

Mean:       ~49.44 ms
Median:     ~46.10 ms
P95:        ~66.14 ms
Throughput: ~20.13 req/s
```

## 25. Quest Untethered Live Classification

Untethered live-classification evidence includes:

```text
quest_cloud_untet_live_classification.jpg
```

This confirms that the Quest application continued to perform camera-to-cloud inference without the tether/casting connection.

## 26. Quest Untethered Frozen-Camera Benchmark

Untethered frozen-camera evidence includes:

```text
quest_cloud_untet_frozen_benchmark.jpg
```

Representative result:

```text
Runs:   100
Mean:   ~47.54 ms
Median: ~45.99 ms
P95:    ~61.43 ms
```

## 27. Quest Untethered End-to-End Camera Benchmark

Untethered end-to-end evidence includes:

```text
quest_cloud_untet_end_to_end.jpg
```

Representative result:

```text
Runs:   100
Mean:   ~70.09 ms
Median: ~72.36 ms
P95:    ~75.90 ms
```

## 28. Quest Untethered Live Throughput

Untethered live-throughput evidence includes:

```text
quest_cloud_untet_live_throughput.jpg
```

Representative completed result:

```text
Completed requests: 192
Time:               ~10.03 s
Throughput:         ~19.15 predictions/s
```

## 29. Quest Untethered OVR Metrics Logs

The untethered Quest cloud telemetry logs are:

```text
quest_cloud_untet_overmetrics.csv
quest_cloud_untet_ovrmetrics2.csv
quest_cloud_untet_ovrmetrics3.csv
```

These logs cover longer untethered benchmark periods and provide device-level telemetry during cloud-assisted execution.

The first filename preserves the original `overmetrics` spelling used in the repository.

## 30. OVR Metrics Fields

The OVR Metrics CSV files contain many Quest runtime fields.

Useful categories for the dissertation may include:

- application frame rate;
- CPU utilization;
- GPU utilization;
- CPU/GPU performance levels;
- application memory;
- available memory;
- GPU memory;
- battery percentage;
- battery temperature;
- frame timing;
- thermal/performance-state indicators; and
- device power telemetry when available.

Not every field needs to be reported.

Only metrics relevant to the performance/resource questions should be extracted for analysis.

## 31. Interpreting Quest Power Telemetry

Quest OVR Metrics power-related fields should be treated carefully.

The raw logs may reflect differences in:

- charging/tether state;
- battery state;
- measurement sign convention;
- device activity; and
- logging conditions.

For this reason, power values should not be interpreted solely from one field without checking the session context and the meaning of the metric.

The repository preserves the raw logs so that the final analysis can be traced back to the original device telemetry.

## 32. Cloud Accuracy Difference

The cloud benchmark uses the same MNIST test data and trained model lineage as the local implementations, but the observed 1,000-image cloud benchmark accuracy is:

```text
957 / 1000
95.70%
```

This should be reported as the observed cloud benchmark result for the tested 1,000-image subset.

It should not be substituted for the full 10,000-image local validation result of 96.63%.

## 33. Evidence Roles

The repository preserves several types of evidence.

### Screenshots

Primary evidence for:

- benchmark results;
- camera predictions;
- latency statistics;
- request throughput; and
- tethered/untethered conditions.

### OVR Metrics CSV files

Supporting device telemetry for Quest execution.

### Windows memory screenshots

Supporting resource evidence for the PC cloud client.

### Videos

Supporting evidence for:

- PC cloud power observations; and
- Internet/network dependence.

Each evidence type serves a different purpose and should not be treated as interchangeable.

## 34. Reproduction Sequence — PC Cloud

A practical PC cloud reproduction sequence is:

```text
[ ] Begin with the existing local PC Unity project
[ ] Configure the Cloud Run endpoint
[ ] Run CloudMNISTSingleTest
[ ] Verify a successful cloud response
[ ] Run CloudMNISTBenchmark
[ ] Capture the controlled benchmark result
[ ] Configure CloudCameraMNISTInference
[ ] Validate live cloud classification
[ ] Run the frozen-camera benchmark
[ ] Run the end-to-end camera benchmark
[ ] Run live cloud throughput
[ ] Repeat key tests in the standalone build
[ ] Capture Windows memory/resource evidence
[ ] Record whole-system power if required
```

## 35. Reproduction Sequence — Quest Cloud

A practical Quest cloud reproduction sequence is:

```text
[ ] Begin with the existing local Quest 3S project
[ ] Use SecondScene
[ ] Configure QuestCloudCameraMNISTInference
[ ] Configure the Cloud Run endpoint
[ ] Build and install the APK
[ ] Confirm Internet access
[ ] Run tethered/cast validation
[ ] Start OVR Metrics recording
[ ] Run controlled cloud benchmark
[ ] Validate cloud camera classification
[ ] Run frozen-camera cloud benchmark
[ ] Run end-to-end cloud benchmark
[ ] Run live cloud throughput
[ ] Save tethered OVR Metrics logs
[ ] Repeat without tether/casting
[ ] Save untethered screenshots and OVR Metrics logs
```

## 36. Repository Evidence Index

### PC cloud

```text
cloud_pc_accuracy_latency_throughput.png
cloud_pc_accuracy_latency_throughput2.png
cloud_pc_Accuracy_latency_throughput_build.png
cloud_pc_classification.png
cloud_pc_end_to_end.png
cloud_pc_end_to_end_build.png
cloud_pc_frozen_benchmark.png
cloud_pc_frozen_benchmark_build.png
cloud_pc_live_classification.png
cloud_pc_live_classification_build.png
cloud_pc_live_throughput.png
cloud_pc_live_throughput_build.png
cloud_pc_memory1.png
cloud_pc_memory2.png
cloud_pc_memory3.png
cloud_pc_memory4.png
cloud_pc_memory5.png
cloud_pc_memory6.png
cloud_pc_memory7.png
```

### Quest cloud — tethered

```text
quest_cloud_accuracy_latency_throughput.png
quest_cloud_accuracy_latency_throughput2.png
quest_cloud_benchmark.png
quest_cloud_benchmark2.png
quest_cloud_benchmark3.png
quest_cloud_benchmark4.png
quest_cloud_end_to_end.png
quest_cloud_end_to_end2.png
quest_cloud_frozen_benchmark.png
quest_cloud_frozen_classification.png
quest_cloud_live_classification.png
quest_cloud_live_throughput.png
quest_cloud_live_throughput2.png
quest_cloud_ovrmetrics.csv
quest_cloud_ovrmetrics2.csv
quest_cloud_ovrmetrics3.csv
quest_cloud_ovrmetrics4.csv
```

### Quest cloud — untethered

```text
quest_cloud_untet_accuracy_latency_throughput.jpg
quest_cloud_untet_benchmark.jpg
quest_cloud_untet_end_to_end.jpg
quest_cloud_untet_frozen_benchmark.jpg
quest_cloud_untet_live_classification.jpg
quest_cloud_untet_live_throughput.jpg
quest_cloud_untet_overmetrics.csv
quest_cloud_untet_ovrmetrics2.csv
quest_cloud_untet_ovrmetrics3.csv
```

### Videos

```text
Cloud Power Unity10-22-202606-36.mp4
Cloud Power 2 Unity10-26-202608-11.mp4
Unity Cloud Internet10-22-202610-13.mp4
```

## 37. Scope

This notebook documents the **cloud-client configuration, benchmark procedure, and evidence** for the MNIST PC and Quest 3S implementations.

It does not attempt to explain why cloud, local PC, local Quest, or FPGA execution performs better or worse.

Those comparisons belong in the dissertation results and discussion, where the cloud measurements are evaluated alongside the corresponding local and FPGA measurements.